# 1. Init — 检查预测 FASTA 并建库

1. 确认 `bin/foldseek`（可从 `0.prepare` 复用或本页下载）
2. 确认 `0.prepare` 已写入 `work/GT_fasta`、`work/DB/{foldseek,mmseqs}_DB`、`work/lable/scop_lookup.tsv`
3. 检查用户放入的 `work/aa2di_fasta/*aa2di.fasta`（见该目录 README）
4. 为预测方法构建 Foldseek DB → `work/DB/{method}_DB`

`work/di2aa_fasta/` 为预留目录，**当前基准不使用**。

共享配置：[`config.py`](config.py)。


In [1]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
import tarfile
import tempfile
from pathlib import Path

from config import (
    AA2DI_FASTA_DIR,
    AA_FASTA,
    BIN_DIR,
    CONDA_ENV,
    DBS_DIR,
    DI2AA_FASTA_DIR,
    FOLDSEEK_BIN,
    FOLDSEEK_GT_DIR,
    FOLDSEEK_TARBALL,
    FOLDSEEK_TMP_DIR,
    FOLDSEEK_URL,
    GITHUB_RUNTIME_URL,
    GT_DI_FASTA,
    GT_FASTA_DIR,
    LEGACY_RUNTIME_TARBALL,
    LABEL_DIR,
    METHODS,
    MMSEQS_BIN,
    MMSEQS_GT_DIR,
    PROJECT_ROOT,
    SCOP_LOOKUP,
    TEMP,
    WORK_DIR,
    db_prefix,
    cleanup_tmp,
    ensure_work_dirs,
    predicted_fasta_names,
    require_project_root,
    scop_cla_path,
    work_ready,
)

ROOT = require_project_root("1.init.ipynb")
assert ROOT == PROJECT_ROOT

SKIP_EXISTING = True
FORCE_REDOWNLOAD_FOLDSEEK = False
FORCE_REINSTALL_WORK = False

ensure_work_dirs()
print("ROOT:", ROOT)
print("work/:", WORK_DIR)
print("GT_fasta:", GT_FASTA_DIR)
print("aa2di_fasta:", AA2DI_FASTA_DIR)
print("di2aa_fasta (reserved):", DI2AA_FASTA_DIR)
print("DB:", DBS_DIR)
print("lable:", LABEL_DIR)
print("foldseek:", FOLDSEEK_BIN)
print("mmseqs:", MMSEQS_BIN)
print("所需 aa2di:", predicted_fasta_names())
print("python:", sys.executable)
print("version:", sys.version.split()[0])


ROOT: /hpcfs/fhome/caihuize/scope40_easy
work/: /hpcfs/fhome/caihuize/scope40_easy/work
GT_fasta: /hpcfs/fhome/caihuize/scope40_easy/work/GT_fasta
aa2di_fasta: /hpcfs/fhome/caihuize/scope40_easy/work/aa2di_fasta
di2aa_fasta (reserved): /hpcfs/fhome/caihuize/scope40_easy/work/di2aa_fasta
DB: /hpcfs/fhome/caihuize/scope40_easy/work/DB
lable: /hpcfs/fhome/caihuize/scope40_easy/work/lable
foldseek: /hpcfs/fhome/caihuize/scope40_easy/bin/foldseek
mmseqs: /hpcfs/fhome/caihuize/scope40_easy/bin/mmseqs
所需 aa2di: ['DB_ESM3_aa2di.fasta', 'DB_ESM3_LoRA_aa2di.fasta', 'DB_ProstT5_translate_aa2di.fasta', 'DB_SaProt_aa2di.fasta']
python: /hpcfs/fhome/caihuize/.conda/envs/ESM3_3Di_5090/bin/python
version: 3.10.20


## 下载 Foldseek → `bin/foldseek`

优先复用 `0.prepare` 已安装的二进制。


In [2]:
def download_foldseek(skip_existing: bool = True, force: bool = False) -> Path:
    """Ensure project-wide binary at bin/foldseek."""
    ensure_work_dirs()
    if not force and skip_existing and FOLDSEEK_BIN.is_file() and os.access(FOLDSEEK_BIN, os.X_OK):
        print(f"⏭️  foldseek 已存在: {FOLDSEEK_BIN}")
        return FOLDSEEK_BIN

    tmp_bin = FOLDSEEK_TMP_DIR / "bin" / "foldseek"
    if not force and tmp_bin.is_file() and os.access(tmp_bin, os.X_OK):
        shutil.copy2(tmp_bin, FOLDSEEK_BIN)
        FOLDSEEK_BIN.chmod(FOLDSEEK_BIN.stat().st_mode | 0o111)
        print(f"✅ 从 tmp 复制 → {FOLDSEEK_BIN}")
        return FOLDSEEK_BIN

    try:
        cpuinfo = Path("/proc/cpuinfo").read_text(encoding="utf-8", errors="ignore")
        if "avx2" not in cpuinfo.lower():
            print("⚠️  /proc/cpuinfo 未看到 avx2；当前包为 linux-avx2，可能无法运行")
    except OSError:
        pass

    print(f"下载 Foldseek:\n  {FOLDSEEK_URL}")
    TEMP.mkdir(parents=True, exist_ok=True)
    if shutil.which("wget"):
        cmd = ["wget", "-O", str(FOLDSEEK_TARBALL), FOLDSEEK_URL]
    elif shutil.which("curl"):
        cmd = ["curl", "-L", "-o", str(FOLDSEEK_TARBALL), FOLDSEEK_URL]
    else:
        raise RuntimeError("需要 wget 或 curl 以下载 foldseek")
    print("[CMD]", " ".join(cmd))
    subprocess.run(cmd, check=True)
    with tarfile.open(FOLDSEEK_TARBALL, "r:gz") as tf:
        tf.extractall(path=TEMP)
    src = FOLDSEEK_TMP_DIR / "bin" / "foldseek"
    if not src.is_file():
        raise FileNotFoundError(f"解压后未找到: {src}")
    shutil.copy2(src, FOLDSEEK_BIN)
    FOLDSEEK_BIN.chmod(FOLDSEEK_BIN.stat().st_mode | 0o111)
    print(f"✅ foldseek: {FOLDSEEK_BIN}")
    return FOLDSEEK_BIN


download_foldseek(skip_existing=SKIP_EXISTING, force=FORCE_REDOWNLOAD_FOLDSEEK)
print("bin:", FOLDSEEK_BIN, "exists=", FOLDSEEK_BIN.is_file())


⏭️  foldseek 已存在: /hpcfs/fhome/caihuize/scope40_easy/bin/foldseek
bin: /hpcfs/fhome/caihuize/scope40_easy/bin/foldseek exists= True


## 准备 `work/` + 检查 `aa2di_fasta`

- **优先**：本地已由 `0.prepare` 写好的 `work/GT_fasta`、`work/DB`、`work/lable/scop_lookup.tsv`
- **备用**：下载旧版 `scope40_runtime.tar.gz` 并迁移到上述布局（`SCOPE40_RUNTIME_URL`）
- **用户预测**：放入 [`work/aa2di_fasta/`](../work/aa2di_fasta/README.md)


In [3]:
def _migrate_legacy_runtime(legacy_root: Path) -> None:
    """Map old scope40_runtime/ layout into work/."""
    ensure_work_dirs()
    mapping = [
        (legacy_root / "FoldseekDB", FOLDSEEK_GT_DIR),
        (legacy_root / "MMseqsDB", MMSEQS_GT_DIR),
    ]
    for src, dst in mapping:
        if not (src / "DB").is_file():
            raise FileNotFoundError(f"旧包缺少: {src / 'DB'}")
        if dst.exists():
            if dst.is_symlink() or dst.is_file():
                dst.unlink()
            else:
                shutil.rmtree(dst)
        shutil.copytree(src, dst)

    for name in ("DB_aa.fasta", "DB_di.fasta"):
        src = legacy_root / "fasta" / name
        if not src.is_file():
            raise FileNotFoundError(src)
        shutil.copy2(src, GT_FASTA_DIR / name)

    # 旧包 metadata/scop_lookup.tsv → work/lable/
    legacy_lookup = legacy_root / "metadata" / "scop_lookup.tsv"
    if not legacy_lookup.is_file():
        raise FileNotFoundError(f"旧包缺少: {legacy_lookup}")
    LABEL_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(legacy_lookup, SCOP_LOOKUP)
    print(f"✅ 已从旧 runtime 迁移 → {WORK_DIR}")
    print(f"   lable: {SCOP_LOOKUP}")


def ensure_work_assets(force: bool = False) -> None:
    ensure_work_dirs()
    if not force and work_ready():
        print(f"⏭️  work/ 已就绪")
        return

    if force:
        for p in (FOLDSEEK_GT_DIR, MMSEQS_GT_DIR):
            if p.exists():
                shutil.rmtree(p) if p.is_dir() and not p.is_symlink() else p.unlink(missing_ok=True)

    # try legacy extract under tmp/scope40_runtime
    legacy = TEMP / "scope40_runtime"
    if (legacy / "FoldseekDB" / "DB").is_file():
        _migrate_legacy_runtime(legacy)
        if work_ready():
            return

    print(f"本地 work/ 不完整，尝试下载旧 runtime:\n  {GITHUB_RUNTIME_URL}")
    if shutil.which("wget"):
        cmd = ["wget", "-O", str(LEGACY_RUNTIME_TARBALL), GITHUB_RUNTIME_URL]
    elif shutil.which("curl"):
        cmd = ["curl", "-L", "-o", str(LEGACY_RUNTIME_TARBALL), GITHUB_RUNTIME_URL]
    else:
        raise RuntimeError("需要 wget 或 curl；或先运行 0.prepare.ipynb")
    print("[CMD]", " ".join(cmd))
    subprocess.run(cmd, check=True)
    if legacy.exists():
        shutil.rmtree(legacy)
    with tarfile.open(LEGACY_RUNTIME_TARBALL, "r:gz") as tf:
        tf.extractall(TEMP)
    if not (legacy / "FoldseekDB" / "DB").is_file():
        # tarball may extract as scope40_runtime/ at TEMP
        raise FileNotFoundError(f"解压后未找到旧 runtime: {legacy}")
    _migrate_legacy_runtime(legacy)
    if not work_ready():
        raise FileNotFoundError("迁移后 work/ 仍不完整，请运行 0.prepare.ipynb")


def check_aa2di_fastas() -> None:
    ensure_work_dirs()
    missing: list[tuple[str, str]] = []
    for label, _key, _engine, filename in METHODS:
        if filename is None:
            continue
        path = AA2DI_FASTA_DIR / filename
        if path.is_file():
            print(f"✅ {label}: {path}")
        else:
            print(f"❌ {label}: 缺少 {path}")
            missing.append((label, filename))
    if missing:
        listing = "\n".join(f"  - {fn}  ({label})" for label, fn in missing)
        raise FileNotFoundError(
            f"请将 AA→3Di 预测 FASTA 复制到 {AA2DI_FASTA_DIR}/ ：\n{listing}\n"
            f"输入参考: {AA_FASTA}\n说明: {AA2DI_FASTA_DIR / 'README.md'}"
        )


def check_environment() -> dict[str, bool]:
    import sys

    status: dict[str, bool] = {}
    print(f"ROOT = {ROOT}")
    print(f"sys.executable     = {sys.executable}")
    print(f"sys.version        = {sys.version.split()[0]}")
    print(f"CONDA_DEFAULT_ENV  = {os.environ.get('CONDA_DEFAULT_ENV', '(unset)')}")
    print(f"期望 conda env     = {CONDA_ENV}")
    kernel_ok = CONDA_ENV in Path(sys.executable).parts
    status["kernel"] = kernel_ok
    if kernel_ok:
        print(f"✅ kernel Python 属于 {CONDA_ENV}")
    else:
        expected_py = Path.home() / ".conda" / "envs" / CONDA_ENV / "bin" / "python"
        print(f"❌ kernel 不是 {CONDA_ENV}（当前 {sys.version.split()[0]}）")
        print(f"   请选择 kernel「{CONDA_ENV}」后 Restart，期望: {expected_py}")

    try:
        import Bio  # noqa: F401
        status["biopython"] = True
        print("✅ biopython")
    except ImportError:
        status["biopython"] = False
        print("❌ biopython 缺失")

    try:
        import pandas  # noqa: F401
        import matplotlib  # noqa: F401
        import numpy  # noqa: F401
        status["pandas_mpl"] = True
        print("✅ pandas / matplotlib / numpy")
    except ImportError as e:
        status["pandas_mpl"] = False
        print(f"❌ {e}")

    status["foldseek"] = FOLDSEEK_BIN.is_file() and os.access(FOLDSEEK_BIN, os.X_OK)
    print(("✅" if status["foldseek"] else "❌") + f" foldseek: {FOLDSEEK_BIN}")

    status["mmseqs"] = MMSEQS_BIN.is_file() and os.access(MMSEQS_BIN, os.X_OK)
    print(("✅" if status["mmseqs"] else "❌") + f" mmseqs: {MMSEQS_BIN}")
    if not status["mmseqs"]:
        print("   → 请先运行 0.prepare.ipynb，安装 bin/mmseqs")

    status["work"] = work_ready()
    print(("✅" if status["work"] else "❌") + f" work GT assets ready")

    status["scop_lookup"] = SCOP_LOOKUP.is_file()
    print(("✅" if status["scop_lookup"] else "❌") + f" {SCOP_LOOKUP}")

    cla = scop_cla_path()
    if cla.is_file():
        print(f"✅ SCOP cla (optional): {cla}")
    else:
        print(f"ℹ️  SCOP cla 未找到（评估用 work/lable/scop_lookup.tsv 即可）: {cla}")

    for label, key, engine, di_name in METHODS:
        if engine == "foldseek" and di_name is None:
            ok = (FOLDSEEK_GT_DIR / "DB").is_file()
            print(("✅" if ok else "❌") + f" {label}: {FOLDSEEK_GT_DIR / 'DB'}")
            status[f"db_{key}"] = ok
        elif engine == "mmseqs" and di_name is None:
            ok = (MMSEQS_GT_DIR / "DB").is_file()
            print(("✅" if ok else "❌") + f" {label}: {MMSEQS_GT_DIR / 'DB'}")
            status[f"db_{key}"] = ok
        else:
            path = AA2DI_FASTA_DIR / di_name
            ok = path.is_file()
            print(("✅" if ok else "❌") + f" {label}: {path}")
            status[f"aa2di_{key}"] = ok

    print(f"ℹ️  di2aa_fasta 预留目录: {DI2AA_FASTA_DIR}（当前评估未使用）")
    return status


ensure_work_assets(force=FORCE_REINSTALL_WORK)
check_aa2di_fastas()
status = check_environment()
missing = [k for k, ok in status.items() if not ok]
if missing:
    import sys

    hint = ""
    if not status.get("kernel", True):
        hint = (
            f"\n当前 kernel 是 {sys.executable}，不是 {CONDA_ENV}。"
            f"请在 Cursor / Jupyter 选择 kernel「{CONDA_ENV}」，Restart Kernel 后再跑本页。"
            "不要 pip install 到现在这个 Python 3.12。"
        )
    raise SystemExit(f"环境未就绪，缺失: {missing}{hint}")
print("\n环境检查通过。")


⏭️  work/ 已就绪
✅ ESM3-3Di: /hpcfs/fhome/caihuize/scope40_easy/work/aa2di_fasta/DB_ESM3_aa2di.fasta
✅ ESM3-LoRA: /hpcfs/fhome/caihuize/scope40_easy/work/aa2di_fasta/DB_ESM3_LoRA_aa2di.fasta
✅ ProstT5 (translate): /hpcfs/fhome/caihuize/scope40_easy/work/aa2di_fasta/DB_ProstT5_translate_aa2di.fasta
✅ SaProt: /hpcfs/fhome/caihuize/scope40_easy/work/aa2di_fasta/DB_SaProt_aa2di.fasta
ROOT = /hpcfs/fhome/caihuize/scope40_easy
sys.executable     = /hpcfs/fhome/caihuize/.conda/envs/ESM3_3Di_5090/bin/python
sys.version        = 3.10.20
CONDA_DEFAULT_ENV  = ESM3_3Di_5090
期望 conda env     = ESM3_3Di_5090
✅ kernel Python 属于 ESM3_3Di_5090
✅ biopython
✅ pandas / matplotlib / numpy
✅ foldseek: /hpcfs/fhome/caihuize/scope40_easy/bin/foldseek
✅ mmseqs: /hpcfs/fhome/caihuize/scope40_easy/bin/mmseqs
✅ work GT assets ready
✅ /hpcfs/fhome/caihuize/scope40_easy/work/lable/scop_lookup.tsv
✅ SCOP cla (optional): /hpcfs/fhome/caihuize/SCOPE/dir.cla.scope.2.08-stable.txt
✅ Foldseek (AA+3Di): /hpcfs/fhome/caihuize

## 构建预测方法数据库

- GT 库已由 `0.prepare` 装入 `work/DB/foldseek_DB`、`mmseqs_DB`（本步只校验）
- 预测方法：`work/GT_fasta/DB_aa.fasta` + `work/aa2di_fasta/*` → `tsv2db` → `work/DB/{method}_DB`


In [4]:
def _seqio():
    try:
        from Bio import SeqIO
    except ImportError as e:
        raise SystemExit("需要 Biopython: pip install biopython") from e
    return SeqIO


def _read_fasta_dict(path: Path) -> dict[str, str]:
    SeqIO = _seqio()
    out: dict[str, str] = {}
    for record in SeqIO.parse(path, "fasta"):
        out[record.id] = str(record.seq)
    return out


def build_tsvs(aa_fasta: Path, di_fasta: Path, tmp_dir: Path) -> None:
    SeqIO = _seqio()
    sequences_aa = _read_fasta_dict(aa_fasta)
    sequences_3di: dict[str, str] = {}
    for record in SeqIO.parse(di_fasta, "fasta"):
        if record.id not in sequences_aa:
            print(f"Warning: ignoring 3Di entry {record.id}, since it is not in the amino-acid FASTA file")
        else:
            sequences_3di[record.id] = str(record.seq).upper()
    for seq_id in sequences_aa:
        if seq_id not in sequences_3di:
            raise SystemExit(f"Error: entry {seq_id} in amino-acid FASTA has no corresponding 3Di string")

    with (tmp_dir / "aa.tsv").open("w", encoding="utf-8") as faa, \
         (tmp_dir / "3di.tsv").open("w", encoding="utf-8") as fdi, \
         (tmp_dir / "header.tsv").open("w", encoding="utf-8") as fh:
        for i, seq_id in enumerate(sequences_aa.keys(), start=1):
            idx = str(i)
            faa.write(f"{idx}\t{sequences_aa[seq_id]}\n")
            fdi.write(f"{idx}\t{sequences_3di[seq_id]}\n")
            fh.write(f"{idx}\t{seq_id}\n")


def run_tsv2db(foldseek_bin: Path, db_out: Path, tmp_dir: Path) -> None:
    cmds = [
        [str(foldseek_bin), "tsv2db", str(tmp_dir / "aa.tsv"), str(db_out), "--output-dbtype", "0"],
        [str(foldseek_bin), "tsv2db", str(tmp_dir / "3di.tsv"), f"{db_out}_ss", "--output-dbtype", "0"],
        [str(foldseek_bin), "tsv2db", str(tmp_dir / "header.tsv"), f"{db_out}_h", "--output-dbtype", "12"],
    ]
    for cmd in cmds:
        print("[CMD]", " ".join(cmd))
        subprocess.run(cmd, check=True)


def build_one(aa_fasta: Path, di_fasta: Path, db_out: Path, skip_existing: bool = True) -> Path:
    if skip_existing and db_out.is_file():
        print(f"⏭️  DB 已存在，跳过: {db_out}")
        return db_out
    if not FOLDSEEK_BIN.is_file():
        raise FileNotFoundError(f"foldseek 不存在: {FOLDSEEK_BIN}")
    db_out.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="scope40_easy_db_") as tmp:
        tmp_dir = Path(tmp)
        build_tsvs(aa_fasta, di_fasta, tmp_dir)
        run_tsv2db(FOLDSEEK_BIN, db_out, tmp_dir)
    print(f"✅ Foldseek DB: {db_out}")
    return db_out


def build_all(skip_existing: bool = True) -> dict[str, Path]:
    ensure_work_dirs()
    if not (FOLDSEEK_GT_DIR / "DB").is_file():
        raise FileNotFoundError(f"缺少 GT Foldseek DB: {FOLDSEEK_GT_DIR / 'DB'}")
    if not (MMSEQS_GT_DIR / "DB").is_file():
        raise FileNotFoundError(f"缺少 GT MMseqs DB: {MMSEQS_GT_DIR / 'DB'}")

    out: dict[str, Path] = {
        "foldseek": FOLDSEEK_GT_DIR / "DB",
        "mmseqs": MMSEQS_GT_DIR / "DB",
    }
    print(f"⏭️  GT foldseek: {out['foldseek']}")
    print(f"⏭️  GT mmseqs:   {out['mmseqs']}")

    for _label, key, engine, di_name in METHODS:
        if di_name is None:
            continue
        if engine != "foldseek":
            raise ValueError(f"预测方法仅支持 foldseek engine，收到: {key}/{engine}")
        out[key] = build_one(
            AA_FASTA,
            AA2DI_FASTA_DIR / di_name,
            db_prefix(key),
            skip_existing=skip_existing,
        )
    return out


dbs = build_all(skip_existing=SKIP_EXISTING)
for key, path in dbs.items():
    print(f"{key:12s} → {path}  exists={path.is_file()}")
print("\nInit 完成。下一步打开 2.a.benchmark.ipynb")


⏭️  GT foldseek: /hpcfs/fhome/caihuize/scope40_easy/work/DB/foldseek_DB/DB
⏭️  GT mmseqs:   /hpcfs/fhome/caihuize/scope40_easy/work/DB/mmseqs_DB/DB
[CMD] /hpcfs/fhome/caihuize/scope40_easy/bin/foldseek tsv2db /tmp/scope40_easy_db_x43r7r_1/aa.tsv /hpcfs/fhome/caihuize/scope40_easy/work/DB/ESM3_DB/DB --output-dbtype 0
tsv2db /tmp/scope40_easy_db_x43r7r_1/aa.tsv /hpcfs/fhome/caihuize/scope40_easy/work/DB/ESM3_DB/DB --output-dbtype 0 

MMseqs Version:           	941cd33ff0771cd2e3f144e3293e22a2b87e9fda
Output database type      	0
Compressed                	0
Verbosity                 	3

Output database type: Aminoacid
Time for merging to DB: 0h 0m 0s 3ms
Time for processing: 0h 0m 0s 18ms
[CMD] /hpcfs/fhome/caihuize/scope40_easy/bin/foldseek tsv2db /tmp/scope40_easy_db_x43r7r_1/3di.tsv /hpcfs/fhome/caihuize/scope40_easy/work/DB/ESM3_DB/DB_ss --output-dbtype 0
tsv2db /tmp/scope40_easy_db_x43r7r_1/3di.tsv /hpcfs/fhome/caihuize/scope40_easy/work/DB/ESM3_DB/DB_ss --output-dbtype 0 

MMseqs V

## 清理临时目录

删除项目根 `tmp/` 与 `work/tmp/`（下载/解压/搜索中间文件）。产物在 `work/` 与 `bin/` 中保留。


In [5]:
cleanup_tmp(also_work_tmp=True)


🧹 已清理: /hpcfs/fhome/caihuize/scope40_easy/tmp
🧹 已清理: /hpcfs/fhome/caihuize/scope40_easy/work/tmp
